# Build the pixel cache and train a control

Closes the M1/M2 pipeline gap: there was no way to train anything. This notebook
builds a 224 px pixel cache using **either our slice selection or the public one**,
then trains a 5-fold control and reports out-of-fold AUC per label plus the gold-58
score.

Covers issues **#2** (cache with our sampling), **#3** (5-fold control), **#4**
(noise floor, via `SEED`), **#6** (the sampling ablation, via `SAMPLING`) and **#7**
(resolution, via `SIZE`).

## How to use it

| run | change | answers |
|---|---|---|
| 1 | `SAMPLING="public"` | the control |
| 2 | `SAMPLING="ours"` | **issue #6** - the thesis |
| 3 | `SAMPLING="public"`, `SEED=1` | **issue #4** - the noise floor |
| 4 | `SIZE=288` | **issue #7** |

Judge run 2 by **how many of the twelve labels move**, not the macro alone. A real
effect lifts about 10 of 12; noise lifts about half.

**Settings:** GPU T4 x2, **internet ON** (it clones the repo for `src/`; the 9-hour
internet-off rule applies only to the submission notebook).

**Cost:** cache ~2-3 h once per `SAMPLING`/`SIZE` combination, then ~1.5 h per
training run. Both resumable.


In [ ]:
# ================================================================= config
SAMPLING      = "ours"      # "ours" -> src/sampling.py, "public" -> fixed budgets
SIZE          = 224         # 224 or 288
N_SLICE       = 9           # per slot; 6 slots -> 54 slices per study
SEED          = 0           # change to 1 for the noise floor (issue #4)
FOLDS         = 5
LABEL_KEY     = "fix"       # column prefix in weak_labels.csv, or "llm" if attached
EPOCHS        = 6
LR            = 1e-4
BATCH_STUDIES = 4
UNFREEZE_LAST = 6           # transformer blocks left trainable
MAX_STUDIES   = 4407
CACHE_BUDGET_H = 7.0
TRAIN_BUDGET_H = 7.0

import os, sys, glob, json, time, math, gc, pathlib, subprocess, collections
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
torch.backends.cudnn.benchmark = True
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
          "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
SLOTS = [("Sagittal",1),("Sagittal",0),("Coronal",1),("Coronal",0),("Axial",1),("Axial",0)]
CROP_MM = 140.0
torch.manual_seed(SEED); np.random.seed(SEED)
if DEV == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"gpu{i}: {p.name} sm_{p.major}{p.minor} {p.total_memory/2**30:.0f} GiB")
print("torch", torch.__version__, "| device", DEV)


In [ ]:
# ============================================== get src/ from the public repo
REPO = "/kaggle/working/rsna-abnorm-det"
if not os.path.isdir(REPO + "/src"):
    found = [p for p in glob.glob("/kaggle/input/*/**/sampling.py", recursive=True)]
    if found:
        REPO = str(pathlib.Path(found[0]).parent.parent)
        print("using attached copy:", REPO)
    else:
        r = subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/Vetri-78640/rsna-abnorm-det", REPO],
                           capture_output=True, text=True)
        print(r.stdout[-300:], r.stderr[-300:])
        assert os.path.isdir(REPO + "/src"), "turn internet ON, or attach the repo as a dataset"
sys.path.insert(0, REPO + "/src")
import sampling, geometry, normalize, folds
print("src loaded from", REPO)


In [ ]:
# ==================================================== discovery (prunes DICOM trees)
SKIP = {"train_series","test_series","train_images","test_images","__pycache__",".git"}
def scan(root, depth=5):
    base = root.rstrip("/").count(os.sep)
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if x not in SKIP]
        if d.count(os.sep) - base >= depth: dirs[:] = []
        for f in files: yield os.path.join(d, f)

ROOT = None
for cand in ["/kaggle/input/rsna-knee-abnormality-detection",
             "/kaggle/input/competitions/rsna-knee-abnormality-detection"]:
    if os.path.exists(cand + "/train.csv"): ROOT = cand; break
assert ROOT, "attach the competition dataset"
INV = [p for r in sorted(glob.glob("/kaggle/input/*/")) for p in scan(r)]
def find(name): return sorted(p for p in INV if os.path.basename(p) == name)

train = pd.read_csv(ROOT + "/train.csv"); train["StudyInstanceUID"] = train.StudyInstanceUID.astype(str)
ser = pd.read_csv(ROOT + "/train_series.csv")
for k in ("StudyInstanceUID","SeriesInstanceUID"): ser[k] = ser[k].astype(str)
SER = {k: v.to_dict("records") for k, v in ser.groupby("StudyInstanceUID")}
gold_mask = train[LABELS].notna().all(axis=1).to_numpy()
print(f"root {ROOT} | {len(train)} studies | {len(ser)} series | {int(gold_mask.sum())} gold")

wl = find("weak_labels.csv")
assert wl, "attach the weak_labels.csv dataset"
W = pd.read_csv(wl[0]); W["StudyInstanceUID"] = W.StudyInstanceUID.astype(str)
W = W.set_index("StudyInstanceUID")
print("weak labels:", wl[0])


In [ ]:
# ============================================================ cache builder
import pydicom, cv2
from pydicom.pixel_data_handlers.util import apply_modality_lut

CACHE = pathlib.Path(f"/kaggle/working/cache_{SAMPLING}_{SIZE}")
CACHE.mkdir(parents=True, exist_ok=True)

def order_and_meta(sdir):
    recs = []
    for f in glob.glob(sdir + "/*.dcm"):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            recs.append({"path": f, "iop": getattr(h, "ImageOrientationPatient", None),
                         "ipp": getattr(h, "ImagePositionPatient", None),
                         "ps": float(getattr(h, "PixelSpacing", [0.5])[0]),
                         "lat": str(getattr(h, "Laterality", "") or ""),
                         "st": float(getattr(h, "SpacingBetweenSlices", 0) or
                                     getattr(h, "SliceThickness", 0) or 0)})
        except Exception:
            pass
    if not recs: return [], 0.5, "", "none"
    # order_slices returns (records, method) - method says how far it fell back
    recs, how = geometry.order_slices(recs)
    ps = float(np.median([r["ps"] for r in recs]))
    return recs, ps, (recs[0]["lat"] or ""), how

def read_px(path):
    d = pydicom.dcmread(path)
    a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a
    return a

def mm_crop_resize(a, ps, size):
    h, w = a.shape
    cpx = min(int(round(CROP_MM / max(ps, 1e-3))), h, w)
    a = a[(h - cpx)//2:(h - cpx)//2 + cpx, (w - cpx)//2:(w - cpx)//2 + cpx]
    return cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)

# The two sampling rules under test.
PUBLIC_K = {("Sagittal",1):18,("Sagittal",0):14,("Coronal",1):12,("Coronal",0):8,
            ("Axial",1):12,("Axial",0):12}
def pick_indices(n, slot, k):
    if SAMPLING == "public":                      # fixed budget over a fixed span
        lo, hi = int(n*0.02), max(int(n*0.98) - 1, int(n*0.02))
        return np.linspace(lo, hi, k).round().astype(int)
    return np.asarray(sampling.sample_indices(n, k, band=sampling.FULL, keep_ends=True))

def pick_series(rows, plane, fs, used):
    cand = [r for r in rows if r["Anatomical_Plane"] == plane
            and int(r["Fat_Suppression"]) == fs and r["SeriesInstanceUID"] not in used]
    return cand[0] if cand else None

ORDER_METHOD = collections.Counter()
LAT_SEEN = collections.Counter()

def build_study(sid):
    rows = SER.get(sid, [])
    vol = np.zeros((len(SLOTS), N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(len(SLOTS), np.uint8)
    used = set()
    for si, (plane, fs) in enumerate(SLOTS):
        r = pick_series(rows, plane, fs, used)
        if r is None: continue
        used.add(r["SeriesInstanceUID"])
        recs, ps, lat, how = order_and_meta(f"{ROOT}/train_series/{sid}/{r['SeriesInstanceUID']}")
        ORDER_METHOD[how] += 1
        if not recs: continue
        idx = pick_indices(len(recs), (plane, fs), N_SLICE)
        arrs = []
        for p in idx:
            try: arrs.append(read_px(recs[int(min(p, len(recs)-1))]["path"]))
            except Exception: arrs.append(None)
        ok = [a for a in arrs if a is not None]
        if not ok: continue
        stack = np.stack([mm_crop_resize(a if a is not None else ok[0], ps, SIZE) for a in arrs])
        stack = normalize.per_series_window(stack)            # per series, not per slice
        # laterality_flips returns a (flip_horizontal, reverse_slice_order) TUPLE.
        # Do not wrap this in a bare except: a wrong unpack would silently skip
        # laterality on every study and still produce a plausible cache.
        side = (lat.upper() or "")[:1]
        flip_h, rev = geometry.laterality_flips(plane, side if side in ("L", "R") else None)
        LAT_SEEN[side or "?"] += 1
        if flip_h: stack = stack[:, :, ::-1]
        if rev: stack = stack[::-1]
        stack = np.ascontiguousarray(stack)
        vol[si] = np.clip(stack, 0, 1) * 255
        mask[si] = 1
    return vol, mask
print("cache builder ready ->", CACHE)


In [ ]:
# ============================================================ build the cache
done = set()
for p in sorted(glob.glob(str(CACHE / "*.npz"))) + \
         [q for q in INV if "/cache_" in q and q.endswith(".npz")]:
    try: done |= {str(s) for s in np.load(p, allow_pickle=True)["ids"]}
    except Exception: pass
order = train.loc[gold_mask, "StudyInstanceUID"].tolist() + \
        train.loc[~gold_mask, "StudyInstanceUID"].sample(frac=1, random_state=0).tolist()
todo = [s for s in order if s not in done and s in SER][:MAX_STUDIES]
print(f"cached {len(done)} | to build {len(todo)}")

t0 = time.time(); buf_v, buf_m, buf_i = [], [], []
shard = len(glob.glob(str(CACHE / "shard_*.npz")))
def flush():
    global buf_v, buf_m, buf_i, shard
    if not buf_v: return
    tmp = CACHE / f"shard_{shard:04d}.tmp.npz"       # .npz last: savez appends it otherwise
    np.savez_compressed(tmp, vols=np.stack(buf_v), masks=np.stack(buf_m),
                        ids=np.array(buf_i, dtype=object))
    os.replace(tmp, CACHE / f"shard_{shard:04d}.npz")
    done.update(buf_i); print(f"  shard {shard} (+{len(buf_i)}, total {len(done)})", flush=True)
    shard += 1; buf_v, buf_m, buf_i = [], [], []

fails = 0
for i, sid in enumerate(todo):
    if time.time() - t0 > CACHE_BUDGET_H * 3600:
        print("cache budget reached; rerun with this output attached to resume"); break
    try:
        v, mk = build_study(sid); buf_v.append(v); buf_m.append(mk); buf_i.append(sid)
    except Exception as e:
        fails += 1
        if fails < 5: print("  fail", sid[:16], type(e).__name__, e)
    if len(buf_v) >= 200: flush(); gc.collect()
    if (i+1) % 200 == 0:
        el = time.time()-t0
        print(f"  {i+1}/{len(todo)} {el/60:.0f}m {el/(i+1):.2f}s/study "
              f"eta {el/(i+1)*(len(todo)-i-1)/60:.0f}m", flush=True)
flush()
print(f"cache done: {len(done)} studies, {fails} failures, "
      f"{sum(p.stat().st_size for p in CACHE.glob('*.npz'))/1e9:.1f} GB")
print("slice ordering method:", dict(ORDER_METHOD))
print("laterality tag seen  :", dict(LAT_SEEN), " ('?' means no Laterality tag)")


In [ ]:
# ================================================================ load cache
shards = sorted(set(glob.glob(str(CACHE / "shard_*.npz")) +
                    [q for q in INV if "/cache_" in q and q.endswith(".npz")]))
V, M, IDS = [], [], []
for p in shards:
    z = np.load(p, allow_pickle=True)
    V.append(z["vols"]); M.append(z["masks"]); IDS += [str(s) for s in z["ids"]]
VOL = np.concatenate(V); MSK = np.concatenate(M); IDS = np.array(IDS); del V, M
_, keep = np.unique(IDS, return_index=True); keep = np.sort(keep)
VOL, MSK, IDS = VOL[keep], MSK[keep], IDS[keep]
print(f"cache {VOL.shape} {VOL.nbytes/1e9:.1f} GB | slots filled {MSK.mean():.2f}")
assert len(IDS) >= 500, "too few studies cached to train"

sub = train.set_index("StudyInstanceUID").loc[IDS]
Y = W.loc[IDS, [f"{LABEL_KEY}::{t}" for t in LABELS]].to_numpy(np.float32)
GOLD = W.loc[IDS, [f"gold::{t}" for t in LABELS]].to_numpy(np.float32)
is_gold = W.loc[IDS, "is_gold"].to_numpy().astype(bool)

clusters = folds.report_clusters(sub.Report.fillna("").tolist())
FOLD = folds.stratified_group_folds(clusters, (Y > 0.5).astype(float), FOLDS, seed=SEED)
print(f"{len(set(clusters.tolist()))} report clusters -> folds {np.bincount(FOLD).tolist()}")


In [ ]:
# ==================================================================== model
import timm
def build_encoder():
    """Try DINOv3/v2 ViT-S from any attached source. Fails loudly: a silent fallback
    to a random or wrong backbone trains fine and logs a plausible score."""
    tried = []
    for name in ["vit_small_patch16_dinov3.lvd1689m", "vit_small_patch14_dinov2.lvd142m",
                 "vit_small_patch16_224.augreg_in21k"]:
        try:
            enc = timm.create_model(name, pretrained=False, num_classes=0, in_chans=3,
                                    img_size=SIZE, dynamic_img_size=True)
            ck = None
            for cand in glob.glob("/kaggle/input/**/*.pt", recursive=True)[:200]:
                if "dino" in cand.lower() or "fold" in cand.lower():
                    ck = cand; break
            loaded = "random init"
            if ck:
                sd = torch.load(ck, map_location="cpu", weights_only=False)
                sd = sd.get("model", sd) if isinstance(sd, dict) else sd
                sd = {k.replace("backbone.", "").replace("enc.", ""): v
                      for k, v in sd.items() if hasattr(v, "shape")}
                miss, unexp = enc.load_state_dict(sd, strict=False)
                hit = len(enc.state_dict()) - len(miss)
                loaded = f"{ck} ({hit}/{len(enc.state_dict())} tensors)"
                if hit < 0.5 * len(enc.state_dict()):
                    tried.append(f"{name}: only {hit} tensors matched from {ck}")
                    continue
            print(f"encoder: {name} | weights: {loaded}")
            return enc, name
        except Exception as e:
            tried.append(f"{name}: {type(e).__name__}: {e}")
    raise RuntimeError("no usable encoder. Tried:\n  " + "\n  ".join(tried))

class SlotModel(nn.Module):
    def __init__(self, enc, dim, n_slot, n_lab=12, drop=0.2):
        super().__init__()
        self.enc = enc
        self.slot = nn.Embedding(n_slot, dim)
        self.norm = nn.LayerNorm(dim)
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n_lab))
        self.clsW = nn.Parameter(torch.zeros(n_lab, dim)); self.clsb = nn.Parameter(torch.zeros(n_lab))
        nn.init.trunc_normal_(self.clsW, std=0.02)
    def forward(self, x, slot_id, valid):
        B, K = x.shape[:2]
        f = self.enc(x.flatten(0, 1)).view(B, K, -1) + self.slot(slot_id)
        h = self.norm(f)
        a = self.att(h).masked_fill(~valid.unsqueeze(-1), -1e4).softmax(1)
        return (torch.einsum("bkn,bkf->bnf", a, h) * self.clsW).sum(-1) + self.clsb

def make_model():
    """Build encoder + head with the freeze applied. Every fold must go through
    this: an earlier version froze only the fold-0 encoder, so later folds
    silently trained every block and were not comparable."""
    e, name = build_encoder()
    blocks = getattr(e, "blocks", [])
    for i, b in enumerate(blocks):
        for q in b.parameters():
            q.requires_grad = i >= len(blocks) - UNFREEZE_LAST
    return SlotModel(e, e.num_features, len(SLOTS)), name, len(blocks)

_m, enc_name, n_blocks = make_model()
DIM = _m.enc.num_features
print(f"dim {DIM} | blocks {n_blocks} | unfrozen {min(UNFREEZE_LAST, n_blocks)} | "
      f"trainable {sum(q.numel() for q in _m.parameters() if q.requires_grad)/1e6:.2f}M")
del _m


In [ ]:
# ==================================================================== train
MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
SLOT_ID = torch.arange(len(SLOTS)).repeat_interleave(N_SLICE)

def batch(idx):
    v = torch.from_numpy(VOL[idx]).float().div_(255.)        # (B, slot, slice, S, S)
    B = v.shape[0]
    v = v.flatten(1, 2).unsqueeze(2).repeat(1, 1, 3, 1, 1)    # (B, K, 3, S, S)
    v = ((v.flatten(0,1) - MEAN) / STD).view(B, -1, 3, SIZE, SIZE)
    valid = torch.from_numpy(MSK[idx]).bool().repeat_interleave(N_SLICE, 1)
    return v, valid

def auc(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p); y, p = y[ok], p[ok]
    n1, n0 = int((y==1).sum()), int((y==0).sum())
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(p).rank().to_numpy()
    return (r[y==1].sum() - n1*(n1+1)/2) / (n1*n0)
macro = lambda y, p: float(np.nanmean([auc(y[:,j], p[:,j]) for j in range(y.shape[1])]))

oof = np.full((len(IDS), 12), np.nan, np.float32); t0 = time.time()
for k in range(FOLDS):
    if time.time() - t0 > TRAIN_BUDGET_H * 3600:
        print("train budget reached"); break
    tr_i = np.flatnonzero((FOLD != k) & ~is_gold); va_i = np.flatnonzero(FOLD == k)
    model = make_model()[0].to(DEV)
    if DEV == "cuda" and torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-2)
    steps = EPOCHS * max(1, len(tr_i)//BATCH_STUDIES)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=steps, pct_start=0.1)
    lossf = nn.BCEWithLogitsLoss(); scaler = torch.amp.GradScaler("cuda", enabled=DEV=="cuda")
    sid = SLOT_ID.to(DEV); step = 0
    for ep in range(EPOCHS):
        model.train(); perm = np.random.default_rng(SEED+ep).permutation(tr_i)
        for a in range(0, len(perm)-BATCH_STUDIES+1, BATCH_STUDIES):
            b = perm[a:a+BATCH_STUDIES]; x, valid = batch(b)
            x, valid = x.to(DEV, non_blocking=True), valid.to(DEV)
            t = torch.from_numpy(Y[b]).to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV=="cuda"):
                loss = lossf(model(x, sid.expand(len(b), -1), valid), t)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if step < steps: sch.step()
            step += 1
        print(f"  fold {k} ep {ep} loss {loss.item():.4f} {(time.time()-t0)/60:.0f}m", flush=True)
    model.eval()
    with torch.no_grad():
        for a in range(0, len(va_i), BATCH_STUDIES):
            b = va_i[a:a+BATCH_STUDIES]; x, valid = batch(b)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV=="cuda"):
                o = model(x.to(DEV), sid.expand(len(b), -1), valid.to(DEV))
            oof[b] = torch.sigmoid(o.float()).cpu().numpy()
    w = np.isfinite(oof).all(1) & ~is_gold
    print(f"fold {k}: OOF macro so far {macro((Y[w]>0.5).astype(float), oof[w]):.4f}", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache() if DEV=="cuda" else None


In [ ]:
# =================================================================== report
w = np.isfinite(oof).all(1)
res = {"sampling": SAMPLING, "size": SIZE, "seed": SEED, "label_key": LABEL_KEY,
       "encoder": enc_name, "n_studies": int(w.sum()), "folds": FOLDS}
yw = (Y[w & ~is_gold] > 0.5).astype(float); pw = oof[w & ~is_gold]
res["oof_macro_weak"] = macro(yw, pw)
print(f"=== OOF against the {LABEL_KEY} key, {int((w & ~is_gold).sum())} studies ===")
print(f"{'label':<18}{'AUC':>8}{'npos':>7}")
per = {}
for j, t in enumerate(LABELS):
    per[t] = auc(yw[:, j], pw[:, j]); print(f"{t:<18}{per[t]:>8.4f}{int(yw[:,j].sum()):>7}")
res["oof_per_label"] = per
print(f"\nmacro {res['oof_macro_weak']:.4f}")

gw = w & is_gold
if gw.sum() >= 20:
    res["gold_macro"] = macro(GOLD[gw], oof[gw]); res["n_gold"] = int(gw.sum())
    print(f"\n=== gold {int(gw.sum())} studies ===\nmacro {res['gold_macro']:.4f}")
    print("the label extractor scores 0.8625 on the gold 58. If the image model is far")
    print("above that, the labels are binding (see wiki/concepts/label-premise.md)")

tag = f"{SAMPLING}_{SIZE}_seed{SEED}_{LABEL_KEY}"
json.dump(res, open(f"/kaggle/working/control_{tag}.json", "w"), indent=1)
np.save(f"/kaggle/working/oof_{tag}.npy", oof)
pd.DataFrame({"StudyInstanceUID": IDS, "fold": FOLD}).to_csv(
    f"/kaggle/working/folds_{tag}.csv", index=False)
print(f"\nwrote control_{tag}.json, oof_{tag}.npy, folds_{tag}.csv")


## Comparing two runs

Put both `control_*.json` side by side. For the ablation (**issue #6**) the number
that matters is **how many of the twelve per-label AUCs improve**, not the macro:

- 10 or more of 12 up: a real effect, the thesis holds.
- 6 or 7 of 12: noise, whatever the macro says.

Compare the gap against the noise floor from the `SEED=1` run (**issue #4**). A
difference smaller than that gap is not a result.
